In [6]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# -------------------------------------------------------------
# 1. Kinematics Parameters (Laboratory Frame Perspective)
# -------------------------------------------------------------
v0 = 1.8          # Horizontal speed of the box and particle
R = 3.5           # Curvature radius
omega = v0 / R    # Cyclotron frequency
n_frames = 90
t = np.linspace(0, 1.2, n_frames)

# Particle trajectory in laboratory frame S
x_particle = R * np.sin(omega * t)
y_particle = R * (1.0 - np.cos(omega * t))

# Box position (moves at constant speed v0 along x)
box_width = 1.8
box_height = 2.0
x_box = v0 * t - box_width / 2.0
y_box = -0.3

# -------------------------------------------------------------
# 2. Figure Setup (Clean & Minimal)
# -------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4.8), dpi=95)

ax.set_xlim(-1.5, 3.5)
ax.set_ylim(-0.8, 2.4)
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.3)

ax.set_title("External Frame View: Moving Box in Magnetic Field", fontsize=11, pad=12)
ax.set_xlabel("x (Laboratory Frame)", fontsize=10)
ax.set_ylabel("y (Laboratory Frame)", fontsize=10)

# Draw uniform Magnetic Field (into page) across the lab
gx, gy = np.meshgrid(np.linspace(-1.0, 3.0, 6), np.linspace(-0.5, 2.0, 5))
ax.scatter(gx, gy, marker='o', s=110, facecolors='none', edgecolors='#0284c7', alpha=0.25, lw=1.2)
ax.scatter(gx, gy, marker='x', s=40, color='#0284c7', alpha=0.25, lw=1.2)
ax.text(3.2, 2.1, r"$\mathbf{B}$ Field ($\otimes$)", color='#0284c7', fontsize=9, ha='right', weight='bold')

# Graphical Elements
# 1. Moving Box (Frame S')
box_patch = Rectangle((x_box[0], y_box), box_width, box_height,
                      edgecolor='#f59e0b', facecolor='#f59e0b', alpha=0.12, lw=2.0)
ax.add_patch(box_patch)

# 2. Inside Observer (Blue marker with label)
observer_dot, = ax.plot([], [], 'o', color='#38bdf8', markersize=8)
observer_label = ax.text(0, 0, 'Observer in Box', fontsize=8, color='#38bdf8', ha='center')

# 3. Charged Particle (Red marker) & Trajectory trail
trail, = ax.plot([], [], color='#ef4444', lw=2.0, alpha=0.6)
particle, = ax.plot([], [], 'o', color='#ef4444', markersize=9)

# -------------------------------------------------------------
# 3. Animation Update Function
# -------------------------------------------------------------
def update(i):
    # Update box position
    box_patch.set_x(x_box[i])
    
    # Update observer position inside the box (at the left corner)
    obs_x = x_box[i] + 0.35
    obs_y = y_box + 0.5
    observer_dot.set_data([obs_x], [obs_y])
    observer_label.set_position((obs_x, obs_y + 0.2))
    
    # Update particle position and trail
    trail.set_data(x_particle[:i+1], y_particle[:i+1])
    particle.set_data([x_particle[i]], [y_particle[i]])
    
    return box_patch, observer_dot, observer_label, trail, particle

# Smooth & slow pacing: 65 ms interval
anim = FuncAnimation(fig, update, frames=n_frames, interval=65, blit=True)

# Save as MP4
anim.save('box_deflection.mp4', writer='ffmpeg', fps=15, extra_args=['-vcodec', 'libx264'])
plt.close(fig)

# Play in Kaggle
HTML(anim.to_html5_video())